# 1. Maximum Likelihood Fitting

We consider a PDF for a continuous random variable $x$ defined as the **sum of a Gaussian $\mathcal{N}(x;\mu,\sigma)$ and an exponential $\mathcal{E}(x;\lambda)$**:

$$f(x;\theta,\mu,\sigma,\lambda) = \theta \cdot \mathcal{N}(x;\mu,\sigma) + (1-\theta) \cdot \mathcal{E}(x;\lambda)$$

where:
- $\theta \in [0,1]$ is the **parameter of interest** (signal fraction)
- $\mu,\sigma$ are the mean and width of the Gaussian (nuisance parameters)
- $\lambda > 0$ is the rate of the exponential (nuisance parameter)

We will:
1. Generate an i.i.d. data sample $x_1, \ldots, x_n$ from this mixture
2. Estimate $\theta$ and the other parameters via **unbinned maximum likelihood**
   - **Case A**: All parameters free (nuisance parameters treated as unknowns)
   - **Case B**: Nuisance parameters fixed to their true values (estimate $\theta$ only)
3. Retrieve and visualise the results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon
from scipy.optimize import minimize

rng = np.random.default_rng(seed=123)

## 1. True parameters and data generation

We fix the **true** parameter values and generate a sample of size $n$.

In [ ]:
# ------- true parameter values -------
theta_true  = 0.3   # signal fraction
mu_true     = 5.0   # Gaussian mean
sigma_true  = 0.5   # Gaussian width
lambda_true = 0.5   # exponential rate  (mean = 1/lambda)

n_samples = 2000    # total number of events

# ------- sampling from the mixture -------
n_signal     = rng.binomial(n_samples, theta_true)
n_background = n_samples - n_signal

x_signal     = rng.normal(mu_true, sigma_true, size=n_signal)
# scipy's expon uses scale = 1/lambda
x_background = rng.exponential(1.0 / lambda_true, size=n_background)

x_data = np.concatenate([x_signal, x_background])
rng.shuffle(x_data)

print(f"Generated {len(x_data)} events  "
      f"({n_signal} signal + {n_background} background)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(x_data, bins=60, range=(0, 14), density=True,
        histtype='stepfilled', alpha=0.4, color='steelblue', label='Data')

xx = np.linspace(0, 14, 500)
pdf_true = (theta_true * norm.pdf(xx, mu_true, sigma_true)
            + (1 - theta_true) * expon.pdf(xx, scale=1.0/lambda_true))
ax.plot(xx, pdf_true, 'r-', lw=2, label='True PDF')

ax.set_xlabel('x')
ax.set_ylabel('Probability density')
ax.set_title('Generated data vs. true PDF')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Unbinned negative log-likelihood

The log-likelihood for the mixture is

$$\ln \mathcal{L}(\theta,\mu,\sigma,\lambda) = \sum_{i=1}^{n} \ln\!\left[\theta \, \mathcal{N}(x_i;\mu,\sigma) + (1-\theta)\, \mathcal{E}(x_i;\lambda)\right]$$

We minimise the **negative** log-likelihood (NLL) numerically.  
We use `logaddexp` for numerical stability when computing $\ln(a+b)$.

In [ ]:
def nll_full(params, data):
    """Negative log-likelihood with all four parameters free."""
    theta, mu, log_sigma, log_lam = params
    sigma = np.exp(log_sigma)   # enforce sigma > 0
    lam   = np.exp(log_lam)     # enforce lambda > 0

    if not (0.0 < theta < 1.0):
        return np.inf

    # log of each component evaluated at data points
    log_sig_term = np.log(theta)       + norm.logpdf(data, mu, sigma)
    log_bkg_term = np.log(1.0 - theta) + expon.logpdf(data, scale=1.0/lam)

    # log(a + b) = log(exp(log_a) + exp(log_b))  via logaddexp
    log_pdf = np.logaddexp(log_sig_term, log_bkg_term)

    return -np.sum(log_pdf)


def nll_fixed_nuisance(theta_val, data, mu, sigma, lam):
    """NLL when nuisance parameters are fixed – function of theta only."""
    if not (0.0 < theta_val < 1.0):
        return np.inf
    log_sig_term = np.log(theta_val)       + norm.logpdf(data, mu, sigma)
    log_bkg_term = np.log(1.0 - theta_val) + expon.logpdf(data, scale=1.0/lam)
    return -np.sum(np.logaddexp(log_sig_term, log_bkg_term))

## 3. Case A – All parameters free

We maximise the likelihood over $(\theta, \mu, \sigma, \lambda)$ simultaneously.

In [ ]:
# Initial guess (close to but not equal to truth)
theta0  = 0.2
mu0     = x_data.mean()          # rough centre of the distribution
sigma0  = 0.6
lam0    = 0.6

x0_full = [theta0, mu0, np.log(sigma0), np.log(lam0)]

result_full = minimize(
    nll_full,
    x0=x0_full,
    args=(x_data,),
    method='Nelder-Mead', #Heuristic method that doesn't require gradients
    options={'xatol': 1e-6, 'fatol': 1e-6, 'maxiter': 50000}
)

theta_hat, mu_hat, log_sigma_hat, log_lam_hat = result_full.x
sigma_hat = np.exp(log_sigma_hat)
lam_hat   = np.exp(log_lam_hat)

# --- Numerical Hessian at the minimum (observed information matrix) ---
# Covariance matrix = H^{-1}; uncertainties = sqrt(diag(H^{-1}))
def numerical_hessian(f, x0, args=(), h=1e-4):
    """Hessian of f at x0 via central finite differences."""
    n = len(x0)
    H = np.zeros((n, n))
    f0 = f(x0, *args)
    for i in range(n):
        for j in range(i, n):
            ei = np.zeros(n); ei[i] = h
            ej = np.zeros(n); ej[j] = h
            if i == j:
                H[i, i] = (f(x0 + ei, *args) - 2*f0 + f(x0 - ei, *args)) / h**2
            else:
                H[i, j] = (f(x0 + ei + ej, *args) - f(x0 + ei - ej, *args)
                           - f(x0 - ei + ej, *args) + f(x0 - ei - ej, *args)) / (4 * h**2)
                H[j, i] = H[i, j]
    return H

H_full   = numerical_hessian(nll_full, result_full.x, args=(x_data,))
cov_full = np.linalg.inv(H_full)

# Uncertainties in θ and μ are direct; σ and λ were fitted in log-space,
# so apply the delta method: err(σ) = σ · err(log σ)
err_theta = np.sqrt(cov_full[0, 0])
err_mu    = np.sqrt(cov_full[1, 1])
err_sigma = sigma_hat * np.sqrt(cov_full[2, 2])   # delta method
err_lam   = lam_hat   * np.sqrt(cov_full[3, 3])   # delta method

print("=== Case A: all parameters free ===")
print(f"  θ̂    = {theta_hat:.4f} ± {err_theta:.4f}   (true: {theta_true})")
print(f"  μ̂    = {mu_hat:.4f} ± {err_mu:.4f}   (true: {mu_true})")
print(f"  σ̂    = {sigma_hat:.4f} ± {err_sigma:.4f}   (true: {sigma_true})")
print(f"  λ̂    = {lam_hat:.4f} ± {err_lam:.4f}   (true: {lambda_true})")
print(f"  NLL  = {result_full.fun:.4f}")
print(f"  Optimiser success: {result_full.success}")


## 4. Case B – Nuisance parameters fixed

Here $\mu, \sigma, \lambda$ are **fixed to their true values** and only $\theta$ is estimated.

In [ ]:
from scipy.optimize import minimize_scalar

eps = 1e-4  # small number to avoid log(0) issues

result_fixed = minimize_scalar(
    nll_fixed_nuisance,
    bounds=(eps, 1 - eps), # theta must be in (0, 1)
    method='bounded', # bounded method for scalar optimization (1D minimization)
    args=(x_data, mu_true, sigma_true, lambda_true)
)

theta_hat_fixed = result_fixed.x

# Uncertainty from the curvature of the NLL at the minimum:
#   Var(θ) ≈ 1 / (d²NLL/dθ²)|_{θ̂}
h = 1e-4
nll_0 = nll_fixed_nuisance(theta_hat_fixed,     x_data, mu_true, sigma_true, lambda_true)
nll_p = nll_fixed_nuisance(theta_hat_fixed + h, x_data, mu_true, sigma_true, lambda_true)
nll_m = nll_fixed_nuisance(theta_hat_fixed - h, x_data, mu_true, sigma_true, lambda_true)
d2nll = (nll_p - 2*nll_0 + nll_m) / h**2
err_theta_fixed = 1.0 / np.sqrt(d2nll)

print("=== Case B: nuisance parameters fixed ===")
print(f"  θ̂    = {theta_hat_fixed:.4f} ± {err_theta_fixed:.4f}   (true: {theta_true})")
print(f"  NLL  = {result_fixed.fun:.4f}")
print(f"  Optimiser success: {result_fixed.success}")


## 5. Visualising the fitted PDFs

In [ ]:
xx = np.linspace(0, 14, 600)

pdf_full = (theta_hat * norm.pdf(xx, mu_hat, sigma_hat)
            + (1 - theta_hat) * expon.pdf(xx, scale=1.0/lam_hat))

pdf_fixed = (theta_hat_fixed * norm.pdf(xx, mu_true, sigma_true)
             + (1 - theta_hat_fixed) * expon.pdf(xx, scale=1.0/lambda_true))

pdf_true_line = (theta_true * norm.pdf(xx, mu_true, sigma_true)
                 + (1 - theta_true) * expon.pdf(xx, scale=1.0/lambda_true))

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(x_data, bins=60, range=(0, 14), density=True,
        histtype='stepfilled', alpha=0.3, color='steelblue', label='Data')
ax.plot(xx, pdf_true_line, 'k--', lw=2,  label='True PDF')
ax.plot(xx, pdf_full,      'r-',  lw=2,  label='MLE fit (all free)')
ax.plot(xx, pdf_fixed,     'g-',  lw=2,  label='MLE fit (nuisance fixed)')

ax.set_xlabel('x')
ax.set_ylabel('Probability density')
ax.set_title('Maximum Likelihood fit to Gaussian + Exponential mixture')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Summary

In [ ]:
import pandas as pd

def fmt(val, err):
    return f"{val:.4f} ± {err:.4f}"

summary_data = {
    "True":                 [theta_true,                              mu_true,                    sigma_true,                   lambda_true],
    "MLE (all free)":       [fmt(theta_hat, err_theta),               fmt(mu_hat, err_mu),        fmt(sigma_hat, err_sigma),    fmt(lam_hat, err_lam)],
    "MLE (nuisance fixed)": [fmt(theta_hat_fixed, err_theta_fixed),   "fixed",                    "fixed",                      "fixed"],
}
df_summary = pd.DataFrame(summary_data, index=["θ", "μ", "σ", "λ"])
df_summary.index.name = "Parameter"
display(df_summary)


**Key observations:**
- Both approaches recover a value of $\theta$ close to the truth given sufficient statistics.

> **Next step:** Confidence intervals and 2-D confidence regions derived from the profile likelihood are covered in [02_confidence_intervals.ipynb](02_confidence_intervals.ipynb).

We will see that:
- When all parameters are free the estimator has larger variance (broader likelihood) because the nuisance parameters absorb some information.
- Fixing nuisance parameters to their true values gives a tighter estimate, at the cost of ignoring any uncertainty on those parameters.

## Exercises

1. **Effect of sample size.** Re-run the fit with $n = 200$, $n = 500$, and $n = 5000$ events. How do the MLE estimates and their spread change? Plot $\hat{\theta}$ as a function of $n$.

2. **Starting-point sensitivity.** Try five very different initial guesses for $\theta_0$ (e.g. 0.01, 0.1, 0.5, 0.9, 0.99) and check whether the optimiser always converges to the same solution. What does this tell you about the shape of the NLL surface?

3. **Binned vs. unbinned likelihood.** Bin the data into 50 equal-width bins and construct the **binned negative log-likelihood** $-\sum_k n_k \ln p_k$ where $p_k = \int_{\text{bin}\,k} f(x;\boldsymbol{\theta})\,dx$. Minimise it and compare the result with the unbinned MLE. Which is more precise, and why?

4. **Signal-fraction near zero.** Set $\theta_{\rm true} = 0.01$ and $n = 2000$. Does the MLE still converge? Check whether the estimate is unbiased (repeat 100 times and look at the distribution of $\hat{\theta}$). Relate your findings to the concept of an *upper limit*.

5. **Extended maximum likelihood.** Replace the normalised PDF with a **Poisson extended likelihood** that also constrains the total number of events. Add $\mu_{\rm tot}$ (expected total events) as a free parameter and verify that you recover $\hat{\mu}_{\rm tot} \approx n_{\rm samples}$.